In [1]:
# Feature Engineering for Flight Delay Prediction
# This notebook transforms cleaned data into ML-ready features

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Load cleaned data from notebook 01
df = pd.read_csv('../data/processed/cleaned_flights.csv', parse_dates=['fl_date'])

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
print(f"\nTarget variable distribution:")
print(df['is_delayed'].value_counts())
print(f"\nClass balance:")
print(df['is_delayed'].value_counts(normalize=True) * 100)

Dataset loaded: 6,982,766 rows × 36 columns
Memory usage: 4.07 GB

Target variable distribution:
is_delayed
0    5587658
1    1395108
Name: count, dtype: int64

Class balance:
is_delayed
0    80.020697
1    19.979303
Name: proportion, dtype: float64


In [2]:
# Extract temporal features from scheduled departure time

print("Creating temporal features...")

# Hour and minute from HHMM format (e.g., 1430 → 14 hours, 30 minutes)
df['dep_hour'] = df['crs_dep_time'] // 100
df['dep_minute'] = df['crs_dep_time'] % 100

# Arrival hour
df['arr_hour'] = df['crs_arr_time'] // 100

# Time of day categories (binary flags)
df['is_morning'] = df['dep_hour'].between(5, 11).astype(int)      # 5 AM - 11 AM
df['is_afternoon'] = df['dep_hour'].between(12, 17).astype(int)   # 12 PM - 5 PM
df['is_evening'] = df['dep_hour'].between(18, 23).astype(int)     # 6 PM - 11 PM
df['is_night'] = df['dep_hour'].between(0, 4).astype(int)         # 12 AM - 4 AM

# Rush hour flags (high traffic times)
df['is_morning_rush'] = df['dep_hour'].isin([6, 7, 8, 9]).astype(int)
df['is_evening_rush'] = df['dep_hour'].isin([16, 17, 18, 19]).astype(int)

# Weekend flag
df['is_weekend'] = df['day_of_week'].isin([6, 7]).astype(int)

# Holiday season (Thanksgiving + Christmas travel)
df['is_holiday_season'] = df['month'].isin([11, 12]).astype(int)

# Cyclical encoding for time (treats time as circular)
# Hour: 23 is close to 0
df['hour_sin'] = np.sin(2 * np.pi * df['dep_hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['dep_hour'] / 24)

# Day of week: Sunday (7) is close to Monday (1)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Month: December (12) is close to January (1)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

print("✓ Temporal features created!")
print(f"\nSample of new features:")
print(df[['crs_dep_time', 'dep_hour', 'is_morning_rush', 'is_weekend', 'hour_sin', 'hour_cos']].head(10))

Creating temporal features...
✓ Temporal features created!

Sample of new features:
   crs_dep_time  dep_hour  is_morning_rush  is_weekend      hour_sin  hour_cos
0          1252        12                0           0  1.224647e-16 -1.000000
1          1015        10                0           0  5.000000e-01 -0.866025
2          1415        14                0           0 -5.000000e-01 -0.866025
3          1650        16                0           0 -8.660254e-01 -0.500000
4          1015        10                0           0  5.000000e-01 -0.866025
5          1410        14                0           0 -5.000000e-01 -0.866025
6           955         9                1           0  7.071068e-01 -0.707107
7          1140        11                0           0  2.588190e-01 -0.965926
8           815         8                1           0  8.660254e-01 -0.500000
9          1300        13                0           0 -2.588190e-01 -0.965926


In [3]:
# Encode categorical variables (airports and carriers)

print("Encoding categorical features...")

# Initialize label encoders
encoder_origin = LabelEncoder()
encoder_dest = LabelEncoder()
encoder_carrier = LabelEncoder()

# Fit and transform
df['origin_encoded'] = encoder_origin.fit_transform(df['origin'])
df['dest_encoded'] = encoder_dest.fit_transform(df['dest'])
df['carrier_encoded'] = encoder_carrier.fit_transform(df['op_unique_carrier'])

print(f"✓ Categorical encoding complete!")
print(f"\nUnique origins: {len(encoder_origin.classes_)}")
print(f"Unique destinations: {len(encoder_dest.classes_)}")
print(f"Unique carriers: {len(encoder_carrier.classes_)}")

print(f"\nSample encoding:")
sample = df[['origin', 'origin_encoded', 'dest', 'dest_encoded', 'op_unique_carrier', 'carrier_encoded']].head(10)
print(sample)

# Save encoders for later use in production
os.makedirs('../data/models', exist_ok=True)
with open('../data/models/label_encoders.pkl', 'wb') as f:
    pickle.dump({
        'origin': encoder_origin,
        'dest': encoder_dest,
        'carrier': encoder_carrier
    }, f)
    
print("\n✓ Encoders saved to ../data/models/label_encoders.pkl")

Encoding categorical features...
✓ Categorical encoding complete!

Unique origins: 348
Unique destinations: 348
Unique carriers: 15

Sample encoding:
  origin  origin_encoded dest  dest_encoded op_unique_carrier  carrier_encoded
0    JFK             173  DTW            98                9E                0
1    MSP             227  CLE            67                9E                0
2    JFK             173  RIC           275                9E                0
3    RIC             275  JFK           173                9E                0
4    DTW              98  MKE           217                9E                0
5    JAX             172  LGA           193                9E                0
6    LGA             193  JAX           172                9E                0
7    CHS              63  LGA           193                9E                0
8    LGA             193  CHS            63                9E                0
9    ITH             168  JFK           173                9

In [4]:
# Create features based on historical performance
# These capture domain knowledge: some airports/carriers are chronically delayed

print("Creating historical delay rate features...")

# Delay rate by origin airport
origin_delay_rate = df.groupby('origin')['is_delayed'].mean()
df['origin_delay_rate'] = df['origin'].map(origin_delay_rate)

# Delay rate by destination airport
dest_delay_rate = df.groupby('dest')['is_delayed'].mean()
df['dest_delay_rate'] = df['dest'].map(dest_delay_rate)

# Delay rate by carrier
carrier_delay_rate = df.groupby('op_unique_carrier')['is_delayed'].mean()
df['carrier_delay_rate'] = df['op_unique_carrier'].map(carrier_delay_rate)

# Delay rate by route (origin-destination pair)
df['route'] = df['origin'] + '_' + df['dest']
route_delay_rate = df.groupby('route')['is_delayed'].mean()
df['route_delay_rate'] = df['route'].map(route_delay_rate)

# Traffic volume features (busy airports have more delays)
origin_volume = df.groupby('origin').size()
df['origin_daily_flights'] = df['origin'].map(origin_volume)

dest_volume = df.groupby('dest').size()
df['dest_daily_flights'] = df['dest'].map(dest_volume)

print("✓ Historical features created!")

print("\n📊 Top 10 airports by delay rate:")
print(origin_delay_rate.sort_values(ascending=False).head(10))

print("\n📊 Top 10 best on-time airports:")
print(origin_delay_rate.sort_values(ascending=True).head(10))

print("\n📊 Carrier on-time performance:")
carrier_performance = carrier_delay_rate.sort_values()
for carrier, rate in carrier_performance.items():
    print(f"{carrier}: {(1-rate)*100:.1f}% on-time")

Creating historical delay rate features...
✓ Historical features created!

📊 Top 10 airports by delay rate:
origin
HTS    0.427184
MGW    0.405405
EWN    0.404255
SPI    0.361905
HGR    0.355401
SCK    0.335283
OTH    0.334311
CKB    0.331683
IAG    0.314667
SMX    0.311321
Name: is_delayed, dtype: float64

📊 Top 10 best on-time airports:
origin
EKO    0.046703
PIH    0.047143
TWF    0.057307
BTM    0.057971
SPN    0.060440
GUM    0.074380
GTR    0.080110
SHR    0.082402
ALW    0.085551
VEL    0.096154
Name: is_delayed, dtype: float64

📊 Carrier on-time performance:
YX: 88.6% on-time
HA: 87.5% on-time
9E: 85.5% on-time
DL: 83.4% on-time
OO: 83.3% on-time
MQ: 81.9% on-time
UA: 81.6% on-time
OH: 80.8% on-time
AS: 80.7% on-time
G4: 79.4% on-time
WN: 77.6% on-time
NK: 75.8% on-time
AA: 75.1% on-time
B6: 74.9% on-time
F9: 73.0% on-time


In [5]:
# Define which features go into the model

# Features we CAN use (available at prediction time)
feature_columns = [
    # Temporal features
    'month', 'day_of_month', 'day_of_week',
    'dep_hour', 'dep_minute', 'arr_hour',
    'is_morning', 'is_afternoon', 'is_evening', 'is_night',
    'is_morning_rush', 'is_evening_rush',
    'is_weekend', 'is_holiday_season',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    
    # Geographic features
    'origin_encoded', 'dest_encoded', 'distance',
    
    # Airline features
    'carrier_encoded',
    
    # Historical features
    'origin_delay_rate', 'dest_delay_rate', 'carrier_delay_rate', 'route_delay_rate',
    'origin_daily_flights', 'dest_daily_flights',
    
    # Flight characteristics
    'crs_elapsed_time'  # Scheduled flight duration
]

# Target variable
target_column = 'is_delayed'

# Create feature matrix and target vector
X = df[feature_columns].copy()
y = df[target_column].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nFeatures being used: {len(feature_columns)}")
print(f"\nFeature list:")
for i, feature in enumerate(feature_columns, 1):
    print(f"{i:2d}. {feature}")

Feature matrix shape: (6982766, 31)
Target vector shape: (6982766,)

Features being used: 31

Feature list:
 1. month
 2. day_of_month
 3. day_of_week
 4. dep_hour
 5. dep_minute
 6. arr_hour
 7. is_morning
 8. is_afternoon
 9. is_evening
10. is_night
11. is_morning_rush
12. is_evening_rush
13. is_weekend
14. is_holiday_season
15. hour_sin
16. hour_cos
17. dow_sin
18. dow_cos
19. month_sin
20. month_cos
21. origin_encoded
22. dest_encoded
23. distance
24. carrier_encoded
25. origin_delay_rate
26. dest_delay_rate
27. carrier_delay_rate
28. route_delay_rate
29. origin_daily_flights
30. dest_daily_flights
31. crs_elapsed_time


In [6]:
# Split data with stratification to maintain class balance

print("Splitting data into train/validation/test sets...")

# First split: 80% train+val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42,
    stratify=y  # Maintains 80/20 class balance in all splits
)

# Second split: 75% train, 25% val (of the 80%)
# This gives us 60% train, 20% val, 20% test overall
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print(f"✓ Data split complete!")
print(f"\nTrain set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val set:   {X_val.shape[0]:,} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:  {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Verify class balance is maintained
print(f"\n📊 Class distribution in each set:")
print(f"Train - Delayed: {y_train.mean()*100:.1f}%")
print(f"Val   - Delayed: {y_val.mean()*100:.1f}%")
print(f"Test  - Delayed: {y_test.mean()*100:.1f}%")

Splitting data into train/validation/test sets...
✓ Data split complete!

Train set: 4,189,659 samples (60.0%)
Val set:   1,396,553 samples (20.0%)
Test set:  1,396,554 samples (20.0%)

📊 Class distribution in each set:
Train - Delayed: 20.0%
Val   - Delayed: 20.0%
Test  - Delayed: 20.0%


In [7]:
# Scale numerical features to have mean=0, std=1
# Neural networks train better with normalized inputs

print("Scaling features...")

# Initialize scaler on training data only
scaler = StandardScaler()
scaler.fit(X_train)

# Transform all three sets using training statistics
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames (optional, but nice for inspection)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_columns)

print("✓ Feature scaling complete!")

# Save scaler for production use
with open('../data/models/feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✓ Scaler saved to ../data/models/feature_scaler.pkl")

# Show before/after example
print("\nExample feature before scaling:")
print(X_train[['distance', 'dep_hour', 'origin_delay_rate']].head(3))
print("\nSame features after scaling:")
print(X_train_scaled[['distance', 'dep_hour', 'origin_delay_rate']].head(3))

Scaling features...
✓ Feature scaling complete!
✓ Scaler saved to ../data/models/feature_scaler.pkl

Example feature before scaling:
         distance  dep_hour  origin_delay_rate
1435044     333.0        15           0.149741
6430539      76.0        12           0.168211
2201894    1956.0         8           0.154135

Same features after scaling:
   distance  dep_hour  origin_delay_rate
0 -0.839841  0.409837          -1.376831
1 -1.270803 -0.202756          -0.868637
2  1.881759 -1.019548          -1.255935


In [8]:
# Save processed datasets for model training

print("Saving processed datasets...")

# Save as CSV
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_val_scaled.to_csv('../data/processed/X_val.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)

y_train.to_csv('../data/processed/y_train.csv', index=False, header=['is_delayed'])
y_val.to_csv('../data/processed/y_val.csv', index=False, header=['is_delayed'])
y_test.to_csv('../data/processed/y_test.csv', index=False, header=['is_delayed'])

print("✓ All datasets saved!")

# Save feature names for later reference
feature_info = {
    'feature_columns': feature_columns,
    'num_features': len(feature_columns),
    'target_column': target_column
}

with open('../data/models/feature_info.pkl', 'wb') as f:
    pickle.dump(feature_info, f)

print("\n✅ Feature engineering complete!")
print(f"\nReady for model training with:")
print(f"  - {len(feature_columns)} input features")
print(f"  - {X_train.shape[0]:,} training samples")
print(f"  - {X_val.shape[0]:,} validation samples")
print(f"  - {X_test.shape[0]:,} test samples")
print(f"\nSaved files:")
print(f"  📁 ../data/processed/X_train.csv")
print(f"  📁 ../data/processed/X_val.csv")
print(f"  📁 ../data/processed/X_test.csv")
print(f"  📁 ../data/models/label_encoders.pkl")
print(f"  📁 ../data/models/feature_scaler.pkl")
print(f"  📁 ../data/models/feature_info.pkl")

Saving processed datasets...
✓ All datasets saved!

✅ Feature engineering complete!

Ready for model training with:
  - 31 input features
  - 4,189,659 training samples
  - 1,396,553 validation samples
  - 1,396,554 test samples

Saved files:
  📁 ../data/processed/X_train.csv
  📁 ../data/processed/X_val.csv
  📁 ../data/processed/X_test.csv
  📁 ../data/models/label_encoders.pkl
  📁 ../data/models/feature_scaler.pkl
  📁 ../data/models/feature_info.pkl
